In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/krupalpatel07/citi-group-dataset-citibank/citigroup.csv


In [3]:
!pip install pykalman -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.1/252.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 7.2 MB/s eta 0:00:00


In [4]:
import plotly.io as pio

pio.renderers.default = 'iframe'

In [5]:
# =====================================================
# 1. IMPORT LIBRARIES
# =====================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import StandardScaler
from pykalman import KalmanFilter

plt.style.use('dark_background')

In [6]:
# =====================================================
# 2. LOAD DATA
# =====================================================
file_path = "/kaggle/input/datasets/krupalpatel07/citi-group-dataset-citibank/citigroup.csv"
df = pd.read_csv(file_path)

In [7]:
# =====================================================
# 3. PREPROCESSING
# =====================================================
df.columns = [c.lower() for c in df.columns]
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.set_index('date', inplace=True)

In [8]:
# =====================================================
# 4. OCEAN HEADER
# =====================================================
from IPython.display import display, HTML

def ocean_header(text):
    display(HTML(f"""
    <div style="
        background: linear-gradient(90deg, #001f3f, #0074D9);
        padding: 20px; border-radius: 14px; margin-top:20px;">
        <h1 style="color:#7FDBFF; text-align:center;">{text}</h1>
    </div>
    """))

ocean_header("🌊 Price Flow Dynamics")

In [9]:
# =====================================================
# 5. PRICE VISUAL
# =====================================================
fig = px.line(df, y='close', title='Citigroup Price')
fig.show()

In [10]:
# =====================================================
# 6. SYNTHETIC COINTEGRATION SETUP
# =====================================================
ocean_header("🔗 Synthetic Cointegration Logic")

# Create synthetic pair using lagged price
x = df['close']
y = df['close'].shift(1)

spread = x - y

df['spread'] = spread

# ADF test
spread_clean = spread.dropna()
adf_result = adfuller(spread_clean)
print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])

fig = px.line(df, y='spread', title='Synthetic Spread')
fig.show()

ADF Statistic: -19.542034246279265
p-value: 0.0


In [11]:
# =====================================================
# 7. KALMAN FILTER SPREAD SMOOTHING
# =====================================================
ocean_header("🧠 Kalman Spread Filter")

kf = KalmanFilter(initial_state_mean=0, n_dim_obs=1)
state_means, _ = kf.filter(df['spread'].fillna(0).values)

df['kalman_spread'] = state_means

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['spread'], name='Raw Spread'))
fig.add_trace(go.Scatter(x=df.index, y=df['kalman_spread'], name='Kalman Smoothed'))
fig.show()

In [12]:
# =====================================================
# 8. Z-SCORE STAT ARB
# =====================================================
ocean_header("📉 Z-Score Arbitrage Engine")

mean = df['kalman_spread'].rolling(20).mean()
std = df['kalman_spread'].rolling(20).std()

df['zscore'] = (df['kalman_spread'] - mean) / std

fig = px.line(df, y='zscore', title='Z-Score')
fig.show()

In [13]:
# =====================================================
# 9. SIGNAL GENERATION
# =====================================================
ocean_header("🎯 Trading Signals")

long_signal = df['zscore'] < -1
short_signal = df['zscore'] > 1

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['close'], name='Price'))
fig.add_trace(go.Scatter(x=df.index[long_signal], y=df['close'][long_signal],
                         mode='markers', name='Long'))
fig.add_trace(go.Scatter(x=df.index[short_signal], y=df['close'][short_signal],
                         mode='markers', name='Short'))
fig.show()

In [14]:
# =====================================================
# 10. REGIME FILTER
# =====================================================
ocean_header("⚡ Volatility Regime Filter")

vol = df['close'].pct_change().rolling(20).std()
df['regime'] = np.where(vol > vol.median(), 'High Vol', 'Low Vol')

fig = px.scatter(df, x=df.index, y='close', color='regime')
fig.show()

In [15]:
# =====================================================
# 11. FINAL INSIGHTS
# =====================================================
ocean_header("📌 Deep Alpha Takeaways")

print("""
1. Synthetic cointegration mimics pairs logic.
2. Kalman filter stabilizes noisy spreads.
3. Z-score provides entry/exit signals.
4. Regime filter avoids unstable periods.
5. This structure forms a base for stat arb strategies.
""")

# =====================================================
# END
# =====================================================



1. Synthetic cointegration mimics pairs logic.
2. Kalman filter stabilizes noisy spreads.
3. Z-score provides entry/exit signals.
4. Regime filter avoids unstable periods.
5. This structure forms a base for stat arb strategies.

